ACGAN generate

In [3]:
import torch
import os
from torchvision.utils import save_image
import torch
import torch.nn as nn
import torch.nn.functional as F

img_size = 112
channels = 3
num_classes = 2
latent_dim = 256

In [7]:


latent_dim = 256
num_classes = 2
n_images = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== 1. 加载模型 =====
generator = Generator().to(device)
generator.load_state_dict(torch.load("generator_2.pth", map_location=device))
generator.eval()

# ===== 2. 输入 =====
noise = torch.randn(n_images, latent_dim).to(device)
labels = torch.randint(0, num_classes, (n_images,)).to(device)

# ===== 3. 生成 =====
with torch.no_grad():
    fake_imgs = generator(noise, labels)

# ===== 4. 反归一化 =====
fake_imgs = (fake_imgs + 1) / 2  # [-1,1] → [0,1]

# ===== 5. 创建文件夹 =====
os.makedirs("generated_images", exist_ok=True)

# ===== 6. 保存 =====
for i in range(n_images):
    save_image(fake_imgs[i], f"generated_images/img_{i}_label_{labels[i].item()}.png")

print("✅ 已保存到 generated_images/")

C:\Users\12656\AppData\Local\Temp\ipykernel_12732\598584771.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  generator.load_state_dict(torch.load("generator_2.pth", map_l

✅ 已保存到 generated_images/


In [6]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + residual)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.label_emb = nn.Embedding(num_classes, 16)

        self.init_size = img_size // 16  # 7
        self.fc = nn.Linear(latent_dim + 16, 256 * self.init_size * self.init_size)

        self.block1 = nn.Sequential(
            nn.BatchNorm2d(256),
            ResBlock(256)
        )

        self.block2 = nn.Sequential(
            nn.Upsample(scale_factor=2),  # 7→14
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            ResBlock(256)
        )

        self.block3 = nn.Sequential(
            nn.Upsample(scale_factor=2),  # 14→28
            nn.Conv2d(256, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            ResBlock(128)
        )

        self.block4 = nn.Sequential(
            nn.Upsample(scale_factor=2),  # 28→56
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            ResBlock(64)
        )

        self.block5 = nn.Sequential(
            nn.Upsample(scale_factor=2),  # 56→112
            nn.Conv2d(64, channels, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        label_embed = self.label_emb(labels)

        x = torch.cat([noise, label_embed], dim=1)

        out = self.fc(x)
        out = out.view(out.size(0), 256, self.init_size, self.init_size)

        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.block4(out)
        img = self.block5(out)

        return img